# Evaluate every model on every test set (STEAD, INSTANCE and the three exploration datasets)

This notebook scores **all 22 trained models and 2 baselines** on **5 test splits**. The scoring code is the same as notebook 07's (`eval_predictor`, cluster bootstrap with 1,000 resamples, evaluation batch 128), so the STEAD and INSTANCE numbers can be compared directly with the earlier results.

| Group | Models | Source |
|---|---|---|
| Authors' pretrained | `pretrained_{None, Linear, LSTM, attention, Transformer}` | repo `trained_models/` |
| Trained on STEAD (paper bottlenecks) | `stead_{None, Linear, LSTM, attention, Transformer}_seed99` | `wdn-runs-all` |
| Trained on STEAD (new variants) | `stead_{attention_PE, attention_LSTM, hybrid, Transformer_fixed}_seed99` | `wdn-runs-all` |
| Trained on INSTANCE | `instance_{None, Linear, LSTM, attention, Transformer}_seed99` | `wdn-runs-all` |
| **Trained on exploration data (final LSTM)** | `vg2d_LSTM_seed99`, `f3_LSTM_seed99`, `str3d_LSTM_seed99` | `wdn-runs-exploration` (notebook 10) |
| Baselines | `baseline_identity`, `baseline_bandpass` | none. The band-pass is 0.5–4.9 Hz for STEAD/INSTANCE; for each exploration set it is the 90 % signal-energy band of its **training** split |

**Test splits:** STEAD (37,400 rows), INSTANCE (6,000), VG2D (6,000), F3 (6,000), STR3D (6,000).

**Outputs** (`results/`, `figures/`, each figure backed by a CSV):

| File | Content |
|---|---|
| `table_all_eval.csv` | every model × test split: median EV (signal/earthquake and noise) with 95 % CIs, mean clipped EV, test loss, correlation, **SNR gain in dB** |
| `table_lstm_transfer_matrix.csv` | the LSTM trained on each of the 5 datasets × the 5 test splits (in-domain and cross-domain) |
| `table_exploration_ranking.csv` | all models ranked on each exploration test split |
| `table_exploration_paired.csv` | in-domain LSTM minus the other models, paired, cluster-bootstrap CI |
| `table_exploration_ev_vs_snr.csv` | EV against input SNR on the exploration splits |
| `check_vs_notebook07.csv` | the STEAD/INSTANCE numbers recomputed here against notebook 07's table |
| `fig_transfer_matrix_lstm` | heat maps of the transfer matrix |
| `fig_exploration_all_models` | every model on each exploration split |
| `fig_exploration_ev_vs_snr`, `fig_exploration_examples_<CODE>`, `fig_exploration_sections`, `fig_exploration_spectra` | supporting figures |

**Kaggle setup:**
- Accelerator: **GPU** (T4 or P100), about 1 h; on CPU about 1.5–2 h. Internet: **ON** (the repo is cloned for the pretrained models).
- Attach:
  1. `wdn-exploration-mixtures` (notebook 09)
  2. `wdn-runs-exploration` (notebook 10)
  3. `wdn-runs-all` (the 14 earlier runs)
  4. `wdn-training-data` (STEAD mixtures)
  5. `instance-mixtures-snr40` (INSTANCE mixtures)
- Run with **Save Version → Save & Run All (Commit)**.

**Resuming:** every model × split result is cached in `results/cache/`. If a session stops, attach the earlier output and run again: cached results are reused.

In [1]:
# SWITCHES
RUN_EVAL    = True        # score every model on every test split (cached per model x split)
RUN_TABLES  = True        # transfer matrix, ranking, paired differences, EV vs SNR, check against notebook 07
RUN_FIGURES = True        # report figures (PDF + PNG)

TEST_SETS   = ["STEAD", "INSTANCE", "VG2D", "F3", "STR3D"]
EXPLORATION = ["VG2D", "F3", "STR3D"]
CATEGORY    = {"STEAD": "earthquake (global)", "INSTANCE": "earthquake (Italy)", "VG2D": "2D marine", "F3": "3D marine", "STR3D": "3D land"}
EVAL_BATCH  = 128         # same as notebook 07 (the paper Transformer depends on the batch composition)
USE_GPU     = True        # float64 on GPU; results equal the CPU numbers to rounding
REUSE_CACHE = True
EV_CLIP     = -1.0
N_BOOT      = 1000
BOOT_SEED   = 12345
BANDPASS_HZ = (0.5, 4.9)  # STEAD / INSTANCE (10 Hz data), as notebook 07
EPOCHS_REQUIRED = 100
INCLUDE_PRETRAINED = True
MAX_TEST_ROWS = None      # e.g. 5000 for a quick trial; None = full test splits
SECTION_N   = 60          # test windows shown side by side in the section figure
EXAMPLE_SEED = 7

COMMIT = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"
STEAD_FILE = "training_datasets_all_snr_40_unshuffled.hdf5"
STEAD_POHA = "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5"
STEAD_QUAKE = "training_datasets_STEAD_waveform_snr_40.hdf5"
INSTANCE_FILE = "training_datasets_INSTANCE_snr40.hdf5"
EXPLORATION_FILE = "training_datasets_{code}.hdf5"
PAPER_BOTTLENECK_NAMES = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECK_NAMES = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")
TRAIN_DOMAINS = ["STEAD", "INSTANCE", "VG2D", "F3", "STR3D"]
RUN_SUFFIX = {"_instance": "INSTANCE", "_vg2d": "VG2D", "_f3": "F3", "_str3d": "STR3D", "_warmup": "STEAD"}
print("switches:", {k: v for k, v in sorted(globals().items()) if k.startswith("RUN_") and isinstance(v, bool)})

switches: {'RUN_EVAL': True, 'RUN_FIGURES': True, 'RUN_TABLES': True}


In [2]:
import os, sys, glob, json, time, re, shutil, subprocess, platform, datetime, hashlib, textwrap
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
TEMP = Path("/kaggle/temp") if ON_KAGGLE else (Path.cwd() / "tmp")
RESULTS, FIGURES, LOGS = WORK / "results", WORK / "figures", WORK / "logs"
for d in (RESULTS, FIGURES, LOGS, TEMP):
    d.mkdir(parents=True, exist_ok=True)

REPO = TEMP / "WaveDecompNet-paper"
SEARCH_ROOTS = [Path("/kaggle/input")] if ON_KAGGLE else []
SEARCH_ROOTS += [WORK, TEMP]

def log(msg):
    stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = "- %s  %s" % (stamp, msg)
    with open(LOGS / "lab_log.md", "a") as f:
        f.write(line + "\n")
    print(line)

def find_all(pattern, roots=None):
    """Every path under the search roots matching a glob pattern, sorted."""
    out = []
    for r in (roots if roots is not None else SEARCH_ROOTS):
        if Path(r).exists():
            out += [Path(p) for p in glob.glob(str(Path(r) / "**" / pattern), recursive=True)]
    out = [p for p in out if not p.name.startswith("._")]
    return sorted(set(out))

def find_one(name, required=True, roots=None):
    hits = find_all(name, roots)
    if not hits:
        if required:
            raise FileNotFoundError(
                "%s not found under %s -- attach the dataset that holds it"
                % (name, [str(r) for r in (roots if roots is not None else SEARCH_ROOTS)]))
        print("NOT FOUND (optional): %s" % name)
        return None
    if len(hits) > 1:
        print("note: %d copies of %s, using %s" % (len(hits), name, hits[0]))
    return hits[0]

def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

print("python      ", platform.python_version())
try:
    import torch
    print("torch       ", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu         ", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch import failed:", exc)
import numpy as np
print("numpy       ", np.__version__)
print("on kaggle   ", ON_KAGGLE)
print("work dir    ", WORK)

if not (REPO / "autoencoder_1D_models_torch.py").exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/yinjiuxun/WaveDecompNet-paper.git", str(REPO)],
                   check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
assert head == COMMIT, "repo is at %s, expected %s" % (head, COMMIT)
print("repo commit ", head)

tt = REPO / "torch_tools.py"
src = tt.read_text()
if "np.Inf" in src:
    tt.write_text(src.replace("np.Inf", "np.inf"))
    print("patched     np.Inf -> np.inf in torch_tools.py")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

log("Part 0 setup: repo %s, python %s" % (head[:12], platform.python_version()))

python       3.13.16
torch        2.11.0+cu130 | cuda available: True
gpu          Tesla T4
numpy        2.1.3
on kaggle    True
work dir     /kaggle/working
repo commit  afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1
patched     np.Inf -> np.inf in torch_tools.py
- 2026-11-27 02:40:26  Part 0 setup: repo afe2536cf29e, python 3.13.16


In [3]:
# Variant bottleneck classes (needed to load the four variant models), same module as notebook 07
MODEL_EXT_SRC = r'''"""Extension bottlenecks for WaveDecompNet.

Repo: yinjiuxun/WaveDecompNet-paper @ afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1

This module adds new bottleneck variants WITHOUT touching
`autoencoder_1D_models_torch.py`, so the Phase 1 / Phase 2 reproduction path
stays byte-identical.

Every bottleneck here is a plain `nn.Module` taking and returning a tensor of
shape (batch, num_steps, num_features) = (B, 75, 64). `SeismogramDecoder.forward`
therefore picks them up through its generic `else: x = self.bottleneck(x)`
branch with no change to the repo file. None of these classes subclasses
`nn.LSTM` or `nn.MultiheadAttention`, which would divert the dispatch.

Variants
--------
attention_PE
    The paper's `attention` bottleneck plus a sinusoidal positional encoding.
    `nn.MultiheadAttention` with query=key=value and no positional information
    is permutation-invariant along time, so the paper's attention bottleneck
    cannot use ordering; position reaches the decoder only through the
    `x + x3` skip connection. The encoding is parameter-free, so this variant
    has EXACTLY the same parameter count as `attention` (111,370) and any
    change in EV cannot be attributed to added capacity.

attention_LSTM
    One bidirectional LSTM layer used as a learned positional embedding,
    followed by multi-head self-attention. This is the repo's own
    `Attention_bottleneck_LSTM`, which `step3` can already select by name but
    which the paper does not report. Reimplemented on `nn.MultiheadAttention`
    to avoid the softmax-axis defect in the repo's hand-rolled
    `DotProductAttention`: `F.softmax(scores, dim=0)` normalises over the
    BATCH axis instead of over keys, so attention weights do not sum to one
    over keys and, at batch size 1, every weight becomes 1.0.

hybrid
    LSTM bottleneck on the earthquake decoder, attention+PE bottleneck on the
    noise decoder. Also already selectable in `step3` and not reported.

Transformer_fixed
    The paper's `Transformer` bottleneck with two defects corrected:
      * `batch_first=True`. The original passes no `batch_first`, so it
        defaults to False and `nn.TransformerEncoder` reads the input
        (batch, steps, features) as (seq, batch, feature) -- it attends across
        the BATCH axis and never along time.
      * `dim_feedforward=256` instead of the 2048 default, a 4x expansion on a
        64-dim latent rather than 32x. The 2048 default accounts for 281,152 of
        the original 640,394 parameters.
    `norm_first=True` makes it a pre-LN block, and the same parameter-free
    positional encoding is applied, since a corrected temporal attention is
    otherwise still permutation-invariant.

The paper's five bottleneck names are deliberately NOT served by
`make_bottlenecks`; they raise `ValueError`. There is no path by which this
module can alter a reproduction run.
"""

import copy
import math

import torch
from torch import nn

__all__ = [
    "SinusoidalPositionalEncoding",
    "AttentionPEBottleneck",
    "AttentionLSTMBottleneck",
    "TransformerFixedBottleneck",
    "make_bottlenecks",
    "NEW_BOTTLENECKS",
    "PAPER_BOTTLENECKS",
]

PAPER_BOTTLENECKS = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECKS = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")


class SinusoidalPositionalEncoding(nn.Module):
    """Parameter-free sinusoidal positional encoding, added to the input.

    Unlike the repo's `PositionalEncoding`, the table is registered as a buffer
    (so it moves with `.to(device)` and is carried by `torch.save(model)`) and
    is built in the model dtype (the repo builds a float32 table inside a
    float64 model).
    """

    def __init__(self, d_model=64, max_len=512, dropout=0.0, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(1, max_len, d_model, dtype=dtype)
        position = torch.arange(max_len, dtype=dtype).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=dtype) * (-math.log(10000.0) / d_model)
        )
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, num_steps, num_features)
        if x.shape[1] > self.pe.shape[1]:
            raise ValueError(
                "sequence length %d exceeds max_len %d" % (x.shape[1], self.pe.shape[1])
            )
        return self.dropout(x + self.pe[:, : x.shape[1], :])


class AttentionPEBottleneck(nn.Module):
    """The paper's `attention` bottleneck + sinusoidal positional encoding.

    Parameter count is identical to the paper's `attention` bottleneck.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, pe_dropout=0.0,
                 max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = SinusoidalPositionalEncoding(d_model, max_len, pe_dropout, dtype)
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x = self.pe(x)
        x, _ = self.attention(x, x, x)
        return x


class AttentionLSTMBottleneck(nn.Module):
    """Bidirectional LSTM as positional embedding, then self-attention.

    Corrected reimplementation of the repo's `Attention_bottleneck_LSTM`.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.lstm = nn.LSTM(
            d_model, d_model // 2, 1, bidirectional=True, batch_first=True, dtype=dtype
        )
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.attention(x, x, x)
        return x


class TransformerFixedBottleneck(nn.Module):
    """Pre-LN transformer encoder block that actually attends along time."""

    def __init__(self, d_model=64, nhead=4, dim_feedforward=256, dropout=0.1,
                 num_layers=1, use_pe=True, max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = (
            SinusoidalPositionalEncoding(d_model, max_len, 0.0, dtype) if use_pe else None
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            dtype=dtype,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False
        )

    def forward(self, x):
        if self.pe is not None:
            x = self.pe(x)
        return self.encoder(x)


def _pair(module):
    """Two bottlenecks with identical initial weights.

    `step3` builds one bottleneck and `copy.deepcopy`s it for the two decoders,
    so both branches start from the same bottleneck weights. That behaviour is
    kept here so the only difference from the paper's runs is the bottleneck
    type.
    """
    return module, copy.deepcopy(module)


def make_bottlenecks(name, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
    """Return (bottleneck_earthquake, bottleneck_noise) for an extension variant.

    Raises ValueError for the paper's five bottleneck names, which must stay on
    the original `step3` code path.
    """
    if name in PAPER_BOTTLENECKS:
        raise ValueError(
            "'%s' is one of the paper's bottlenecks and must be built by the "
            "original step3 code, not by wdn_model_ext." % name
        )

    if name == "attention_PE":
        return _pair(AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "attention_LSTM":
        return _pair(AttentionLSTMBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "Transformer_fixed":
        return _pair(TransformerFixedBottleneck(d_model, nhead, dtype=dtype))

    if name == "hybrid":
        # Repo intent (step3 'hybrid'): LSTM for earthquake, attention for noise.
        # The LSTM matches the paper's `LSTM` bottleneck (2 layers, bidirectional).
        bottleneck_earthquake = nn.LSTM(
            d_model, d_model // 2, 2, bidirectional=True, batch_first=True, dtype=dtype
        )
        bottleneck_noise = AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype)
        return bottleneck_earthquake, bottleneck_noise

    raise NameError(
        "unknown bottleneck '%s'; extension variants are %s" % (name, list(NEW_BOTTLENECKS))
    )
'''

import importlib
for _dest in (REPO, WORK):
    (_dest / "wdn_model_ext.py").write_text(MODEL_EXT_SRC)
importlib.invalidate_caches()
import wdn_model_ext as _wme
importlib.reload(_wme)
print("wdn_model_ext.py sha256[:16] =", sha16(REPO / "wdn_model_ext.py"), "| variants:", list(_wme.NEW_BOTTLENECKS))

wdn_model_ext.py sha256[:16] = 65dea6e1eb352830 | variants: ['attention_PE', 'attention_LSTM', 'hybrid', 'Transformer_fixed']


In [4]:
# Helper Functions 

import numpy as np
import pandas as pd
import h5py

READ_BLOCK = 20000 
BOOT_Q = (2.5, 97.5)


def read_csv_keep(path):
    """pandas reads the string 'None' (a bottleneck name) as missing by default."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])


# Metrics
def ev_raw(y, p):
    """Explained variance per row and component. y, p: (n, T, 3) -> (n, 3).

    Mirrors sklearn.explained_variance_score(multioutput='raw_values'),
    including its handling of a zero denominator.
    """
    d = y - p
    num = d.var(axis=1)
    den = y.var(axis=1)
    out = np.ones_like(den)
    nz_den, nz_num = den != 0, num != 0
    v = nz_den & nz_num
    out[v] = 1.0 - num[v] / den[v]
    out[nz_num & ~nz_den] = 0.0
    return out

def clipped_mean(ev, lo=None):
    """Row-wise mean of component EV after clipping to [lo, 1]."""
    lo = EV_CLIP if lo is None else lo
    return np.clip(ev, lo, 1.0).mean(axis=1)

def row_corr(y, p):
    """Pearson correlation per row over the flattened (T x 3) vector."""
    a = y.reshape(len(y), -1)
    b = p.reshape(len(p), -1)
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    na = np.sqrt((a * a).sum(axis=1))
    nb = np.sqrt((b * b).sum(axis=1))
    den = na * nb
    out = np.zeros(len(a))
    ok = den > 0
    out[ok] = (a[ok] * b[ok]).sum(axis=1) / den[ok]
    return out

# Cluster bootstrap
def make_boot_indices(clusters, n_boot=None, seed=None):
    """Row indices for `n_boot` bootstrap resamples of whole earthquakes."""
    n_boot = N_BOOT if n_boot is None else n_boot
    seed = BOOT_SEED if seed is None else seed
    clusters = np.asarray(clusters)
    uniq, inv = np.unique(clusters, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    sorted_inv = inv[order]
    lo = np.searchsorted(sorted_inv, np.arange(len(uniq)), side="left")
    hi = np.searchsorted(sorted_inv, np.arange(len(uniq)), side="right")
    groups = [order[lo[k]:hi[k]] for k in range(len(uniq))]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        out.append(np.concatenate([groups[k] for k in pick]).astype(np.int32))
    print("bootstrap: %d resamples over %d clusters (%d rows)"
          % (n_boot, len(uniq), len(clusters)))
    return out

def boot_ci(stat_fn, boot_idx):
    """(lo, hi) percentile CI of stat_fn evaluated on each resample."""
    vals = np.fromiter((stat_fn(ix) for ix in boot_idx), dtype=np.float64, count=len(boot_idx))
    return float(np.percentile(vals, BOOT_Q[0])), float(np.percentile(vals, BOOT_Q[1]))

# Predictors
def bandpass_filter(x, lo_hi=None, fs=10.0, order=4):
    from scipy.signal import butter, sosfiltfilt
    lo, hi = BANDPASS_HZ if lo_hi is None else lo_hi
    sos = butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, x, axis=1)

def make_predictor(kind, model=None):
    """Uniform interface: f(x_batch (b,T,3)) -> (eq, noise), both (b,T,3)."""
    if kind == "identity":
        return lambda xb: (xb.copy(), xb.copy())
    if kind == "bandpass":
        def f(xb):
            eq = np.ascontiguousarray(bandpass_filter(xb))
            return eq, xb - eq
        return f

    import torch
    def f(xb):
        with torch.no_grad():
            t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(xb, -1, 1)))
            a, b = model(t.to(torch.float64))
            return (np.moveaxis(a.detach().numpy(), 1, -1),
                    np.moveaxis(b.detach().numpy(), 1, -1))
    return f

def load_model(path):
    import torch
    m = torch.load(str(path), map_location="cpu", weights_only=False)
    m.eval()
    m.to(torch.float64)
    return m

def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

# Scoring
def eval_predictor(pred_fn, X, Y, batch=None, keep_rows=()):
    """Per-row metrics for one predictor on fixed, ordered batches.

    Returns a dict of arrays plus `waveforms` for the row positions in
    `keep_rows` (used by the example-waveform figure).
    """
    batch = EVAL_BATCH if batch is None else batch
    n = len(X)
    out = {k: np.empty(n) for k in ("mse_eq", "mse_noise", "corr_eq", "resid")}
    out["ev_eq"] = np.empty((n, X.shape[2]))
    out["ev_noise"] = np.empty((n, X.shape[2]))
    keep = set(int(k) for k in keep_rows)
    waveforms = {}
    for s in range(0, n, batch):
        e = min(s + batch, n)
        xb, yb = X[s:e], Y[s:e]
        nb = xb - yb
        peq, pnz = pred_fn(xb)
        out["ev_eq"][s:e] = ev_raw(yb, peq)
        out["ev_noise"][s:e] = ev_raw(nb, pnz)
        out["mse_eq"][s:e] = ((yb - peq) ** 2).mean(axis=(1, 2))
        out["mse_noise"][s:e] = ((nb - pnz) ** 2).mean(axis=(1, 2))
        out["corr_eq"][s:e] = row_corr(yb, peq)
        denom = (xb ** 2).mean(axis=(1, 2))
        resid = ((peq + pnz - xb) ** 2).mean(axis=(1, 2))
        out["resid"][s:e] = np.where(denom > 0, resid / np.maximum(denom, 1e-300), 0.0)
        for k in keep:
            if s <= k < e:
                waveforms[k] = dict(x=xb[k - s].copy(), y=yb[k - s].copy(), peq=peq[k - s].copy(), pnz=pnz[k - s].copy())
    out["ev_eq_clipped"] = clipped_mean(out["ev_eq"])
    out["ev_noise_clipped"] = clipped_mean(out["ev_noise"])
    out["waveforms"] = waveforms
    return out


# Test splits
def stead_split_indices(n):
    """Rebuild step3's split on row indices: train_size=0.6/seed 13, then 50/50/seed 20."""
    from sklearn.model_selection import train_test_split
    idx = np.arange(n)
    tr, rest = train_test_split(idx, train_size=0.6, random_state=13)
    va, te = train_test_split(rest, test_size=0.5, random_state=20)
    return tr, va, te

def load_rows(h5path, row_idx, x_key="X_train", y_key="Y_train"):
    """Read the given rows of X and Y in file order, block by block."""
    ri = np.sort(np.asarray(row_idx))
    with h5py.File(h5path, "r") as f:
        X, Y = f[x_key], f[y_key]
        n, T, C = X.shape
        Xo = np.empty((len(ri), T, C), dtype=np.float64)
        Yo = np.empty_like(Xo)
        w = 0
        for s in range(0, n, READ_BLOCK):
            e = min(s + READ_BLOCK, n)
            m = ri[(ri >= s) & (ri < e)]
            if len(m) == 0:
                continue
            xb, yb = X[s:e], Y[s:e]
            Xo[w:w + len(m)] = xb[m - s]
            Yo[w:w + len(m)] = yb[m - s]
            w += len(m)
    assert w == len(ri)
    return ri, Xo, Yo

# Run discovery
def parse_run_folder(name):
    """'Branch_Encoder_Decoder_<bottleneck><suffixes>' -> (bottleneck, trained_on, seed, kind)."""
    prefix = "Branch_Encoder_Decoder_"
    if not name.startswith(prefix):
        return None
    rest = name[len(prefix):]
    kind = "smoke" if "_smoke" in rest else "full"
    trained_on = "INSTANCE" if "_instance" in rest else "STEAD"
    b, seed = rest, 99
    m = re.search(r"_seed(\d+)$", b)
    if m:
        seed = int(m.group(1))
        b = b[:m.start()]
    for suf in ("_instance", "_warmup", "_smoke"):
        if b.endswith(suf):
            b = b[:-len(suf)]
    b = b.replace("_instance", "").replace("_warmup", "").replace("_smoke", "")
    return b, trained_on, seed, kind

def model_id(bottleneck, trained_on, seed):
    return "%s_%s_seed%d" % (trained_on.lower(), bottleneck, seed)

def discover_runs(require_epochs=None):
    """Every finished training run under the search roots, keyed by model id."""
    require_epochs = EPOCHS_REQUIRED if require_epochs is None else require_epochs
    runs = {}
    for pth in find_all("*_Model.pth"):
        if REPO in pth.parents:
            continue
        parsed = parse_run_folder(pth.parent.name)
        if parsed is None:
            continue
        b, trained_on, seed, kind = parsed
        if kind == "smoke":
            continue
        hist = sorted(pth.parent.glob("*_Training_history.hdf5"))
        epochs = None
        if hist:
            try:
                with h5py.File(hist[0], "r") as f:
                    epochs = int(len(f["loss"][:]))
            except Exception:
                epochs = None
        if require_epochs and (epochs is None or epochs < require_epochs):
            print("skip %-44s (epochs=%s)" % (pth.parent.name, epochs))
            continue
        mid = model_id(b, trained_on, seed)
        if mid in runs:
            continue
        info = {}
        for jf in ("run_info.json",):
            jp = pth.parent / jf
            if jp.exists():
                try:
                    info.update(json.loads(jp.read_text()))
                except Exception:
                    pass
        runs[mid] = dict(model_id=mid, bottleneck=b, trained_on=trained_on, seed=seed, path=pth, folder=pth.parent, epochs=epochs,
                         is_new=b in NEW_BOTTLENECK_NAMES, runtime_s=info.get("runtime_s"), run_info=info, kind=("new" if b in NEW_BOTTLENECK_NAMES else "paper"))

    if globals().get("INCLUDE_PRETRAINED", False):
        for pth in sorted((REPO / "trained_models").glob("Branch_Encoder_Decoder_*/*_Model.pth")):
            b = pth.parent.name[len("Branch_Encoder_Decoder_"):]
            mid = "pretrained_%s" % b
            runs[mid] = dict(model_id=mid, bottleneck=b, trained_on="AUTHORS", seed="", path=pth, folder=pth.parent, 
                             epochs=None, is_new=False, runtime_s=None, run_info={}, kind="pretrained")
    return dict(sorted(runs.items()))

print("helpers ready")

# ---- notebook 07's STEAD / INSTANCE test-set loader (unchanged)
def load_test_set(which):
    """Test rows, targets, per-row metadata and bootstrap cluster labels."""
    if which == "STEAD":
        path = find_one(STEAD_FILE)
        with h5py.File(path, "r") as f:
            n_all = f["X_train"].shape[0]
            src_all = f["source_index"][:]
            n_poha = int(f.attrs.get("n_poha", 87000))
        _, _, te = stead_split_indices(n_all)
        te = np.sort(te)
        if MAX_TEST_ROWS:
            te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
        rows, X, Y = load_rows(path, te)
        src = src_all[rows]
        meta = pd.DataFrame(dict(file_row=rows, source_index=src))
        meta["source"] = np.where(src < n_poha, "POHA", "STEAD")
        for col in ("snr", "shift", "quake_energy_frac", "earthquake_name"):
            meta[col] = np.nan if col != "earthquake_name" else ""
        pp, qp = find_one(STEAD_POHA, required=False), find_one(STEAD_QUAKE, required=False)
        if pp is not None and qp is not None:
            def grab(p):
                with h5py.File(p, "r") as f:
                    d = {}
                    for col in ("snr", "shift", "quake_energy_frac"):
                        d[col] = f[col][:] if col in f else None
                    nm = f["earthquake_name"][:] if "earthquake_name" in f else None
                    d["earthquake_name"] = (
                        np.array([s.decode() if isinstance(s, bytes) else str(s) for s in nm])
                        if nm is not None else None)
                return d
            P, Q = grab(pp), grab(qp)
            is_p = src < n_poha
            ip, iq = src[is_p], src[~is_p] - n_poha
            for col in ("snr", "shift", "quake_energy_frac"):
                if P[col] is not None and Q[col] is not None:
                    v = np.full(len(src), np.nan)
                    v[is_p] = P[col][ip]
                    v[~is_p] = Q[col][iq]
                    meta[col] = v
            if P["earthquake_name"] is not None and Q["earthquake_name"] is not None:
                nm = np.empty(len(src), dtype=object)
                nm[is_p] = P["earthquake_name"][ip]
                nm[~is_p] = Q["earthquake_name"][iq]
                meta["earthquake_name"] = nm
                clusters = np.array(["%s:%s" % (s, n) for s, n in
                                     zip(meta["source"], meta["earthquake_name"])])
            else:
                clusters = src.astype(str)
        else:
            print("WARNING: intermediate STEAD files not attached -- no per-row SNR, "
                  "and the bootstrap clusters fall back to source_index "
                  "(which under-counts correlation between rows of one earthquake).")
            clusters = src.astype(str)
    else:
        path = find_one(INSTANCE_FILE)
        with h5py.File(path, "r") as f:
            te = np.where(f["split_id"][:] == 2)[0]
            te = np.sort(te)
            if MAX_TEST_ROWS:
                te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
            cols = {}
            for col in ("snr", "shift", "quake_energy_frac", "source_id",
                        "source_magnitude", "event_snr_mean_db", "p_time_in_window_s",
                        "station_channels", "earthquake_name"):
                if col in f:
                    v = f[col][:][te]
                    if v.dtype.kind in "SO":
                        v = np.array([s.decode() if isinstance(s, bytes) else str(s) for s in v])
                    cols[col] = v
        rows, X, Y = load_rows(path, te)
        cols["file_row"] = rows
        meta = pd.DataFrame(cols)
        clusters = (meta["source_id"].astype(str).values if "source_id" in meta
                    else rows.astype(str))
    print("%s test split: %d rows, X %s" % (which, len(X), X.shape))
    print("  distinct bootstrap clusters:", len(np.unique(clusters)))
    return dict(name=which, path=path, rows=rows, X=X, Y=Y, meta=meta, clusters=clusters)



# ---- additions for notebook 11 -----------------------------------------------------------------
import torch
DEVICE = torch.device("cuda:0" if (USE_GPU and torch.cuda.is_available()) else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

def load_model(path):
    # notebook 07's loader, on DEVICE
    m = torch.load(str(path), map_location=DEVICE, weights_only=False)
    m.eval()
    m.to(torch.float64)
    return m

def make_predictor(kind, model=None, band=None, fs=10.0):
    # notebook 07's interface, f(x (b,T,3)) -> (signal, noise); band-pass with a per-test-set band; model on DEVICE
    if kind == "identity":
        return lambda xb: (xb.copy(), xb.copy())
    if kind == "bandpass":
        def f(xb):
            eq = np.ascontiguousarray(bandpass_filter(xb, lo_hi=band, fs=fs))
            return eq, xb - eq
        return f
    def f(xb):
        with torch.no_grad():
            t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(xb, -1, 1))).to(DEVICE, torch.float64)
            a, b = model(t)
            return np.moveaxis(a.detach().cpu().numpy(), 1, -1), np.moveaxis(b.detach().cpu().numpy(), 1, -1)
    return f

def snr_gain_db(X, Y, pred_fn, batch=None):
    # per row: output SNR - input SNR = 10 log10(||noise||^2 / ||signal - predicted signal||^2)
    batch = EVAL_BATCH if batch is None else batch
    g = np.empty(len(X)); sin = np.empty(len(X))
    for s in range(0, len(X), batch):
        xb, yb = X[s:s + batch], Y[s:s + batch]
        peq, _ = pred_fn(xb)
        n2 = ((xb - yb) ** 2).sum(axis=(1, 2)); e2 = ((yb - peq) ** 2).sum(axis=(1, 2)); y2 = (yb ** 2).sum(axis=(1, 2))
        g[s:s + batch] = 10 * np.log10((n2 + 1e-30) / (e2 + 1e-30))
        sin[s:s + batch] = 10 * np.log10((y2 + 1e-30) / (n2 + 1e-30))
    return g, sin

def parse_run_folder2(name):
    # 'Branch_Encoder_Decoder_<bottleneck><suffix>[_seedN]' -> (bottleneck, trained_on, seed); None for smoke runs / other folders
    prefix = "Branch_Encoder_Decoder_"
    if not name.startswith(prefix) or "_smoke" in name:
        return None
    rest, seed = name[len(prefix):], 99
    m = re.search(r"_seed(\d+)$", rest)
    if m:
        seed, rest = int(m.group(1)), rest[:m.start()]
    for suf, dom in RUN_SUFFIX.items():
        if rest.endswith(suf):
            return rest[:-len(suf)], dom, seed
    return None

def discover_runs2():
    runs = {}
    for pth in find_all("*_Model.pth"):
        if REPO in pth.parents:
            continue
        parsed = parse_run_folder2(pth.parent.name)
        if parsed is None:
            continue
        b, dom, seed = parsed
        hist = sorted(pth.parent.glob("*_Training_history.hdf5")); epochs = None
        if hist:
            try:
                with h5py.File(hist[0], "r") as f:
                    epochs = int(len(f["loss"][:]))
            except Exception:
                epochs = None
        if epochs is None or epochs < EPOCHS_REQUIRED:
            print("skip %-48s (epochs=%s)" % (pth.parent.name, epochs)); continue
        mid = "%s_%s_seed%d" % (dom.lower(), b, seed)
        if mid in runs:
            continue
        info = json.loads((pth.parent / "run_info.json").read_text()) if (pth.parent / "run_info.json").exists() else {}
        kind = "exploration" if dom in EXPLORATION else ("new" if b in NEW_BOTTLENECK_NAMES else "paper")
        runs[mid] = dict(model_id=mid, bottleneck=b, trained_on=dom, seed=seed, path=pth, epochs=epochs, kind=kind, runtime_s=info.get("runtime_s"))
    if INCLUDE_PRETRAINED:
        for pth in sorted((REPO / "trained_models").glob("Branch_Encoder_Decoder_*/*_Model.pth")):
            b = pth.parent.name[len("Branch_Encoder_Decoder_"):]
            runs["pretrained_%s" % b] = dict(model_id="pretrained_%s" % b, bottleneck=b, trained_on="AUTHORS", seed="", path=pth, epochs=None,
                                             kind="pretrained", runtime_s=None)
    return dict(sorted(runs.items()))

def signal_band(path, dt, n_rows=2000, lo_q=0.05, hi_q=0.95):
    # 90 % signal-energy band of the TRAINING split (never the test split), for the band-pass baseline
    with h5py.File(path, "r") as f:
        tr = np.where(f["split_id"][:] == 0)[0]
        pick = np.sort(tr[np.linspace(0, len(tr) - 1, min(n_rows, len(tr))).astype(int)])
        Yt = f["Y_train"][pick]
    P = (np.abs(np.fft.rfft(Yt, axis=1)) ** 2).mean(axis=(0, 2)); fr = np.fft.rfftfreq(Yt.shape[1], dt)
    c = np.cumsum(P) / P.sum(); ny = 0.5 / dt
    lo = max(float(fr[np.searchsorted(c, lo_q)]), fr[1]); hi = min(float(fr[np.searchsorted(c, hi_q)]), 0.95 * ny)
    return (lo, hi) if hi > lo * 1.05 else (fr[1], 0.95 * ny)

def load_exploration_test(code_):
    path = find_one(EXPLORATION_FILE.format(code=code_))
    with h5py.File(path, "r") as f:
        te = np.sort(np.where(f["split_id"][:] == 2)[0])
        if MAX_TEST_ROWS:
            te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
        dt = float(f.attrs["dt_s"]); cols = {}
        for col in ("snr", "source_id", "earthquake_name", "noise_name", "signal_start_sample", "quake_energy_frac"):
            if col in f:
                v = f[col][:][te]
                if v.dtype.kind in "SO":
                    v = np.array([s.decode() if isinstance(s, bytes) else str(s) for s in v])
                cols[col] = v
    rows, X, Y = load_rows(path, te)
    cols["file_row"] = rows
    meta = pd.DataFrame(cols)
    band = signal_band(path, dt)
    print("%s test split: %d rows, X %s, dt %.4g ms, band-pass baseline %.1f-%.1f Hz (training-split signal band)"
          % (code_, len(X), X.shape, dt * 1e3, band[0], band[1]))
    clusters = meta["source_id"].astype(str).values          # spatial block of the signal
    print("  distinct bootstrap clusters (spatial blocks):", len(np.unique(clusters)))
    return dict(name=code_, path=path, rows=rows, X=X, Y=Y, meta=meta, clusters=clusters, fs=1.0 / dt, band=band, dt=dt)

def load_any_test(which):
    if which in EXPLORATION:
        return load_exploration_test(which)
    TS = load_test_set(which)
    TS.update(fs=10.0, band=BANDPASS_HZ, dt=0.1)
    return TS

print("notebook 11 helpers ready")

helpers ready
device: cuda:0 Tesla T4
notebook 11 helpers ready


## Scoring

In [5]:
CACHE = RESULTS / "cache"; CACHE.mkdir(exist_ok=True)
SNR_EDGES = np.logspace(-1, 2, 13)
SUMMARY, SNR_ROWS = [], []

PRIOR_CACHE = {p.name: p for p in find_all("r11__*.npz")}        # results cached by an earlier session (attached output)

def cached(mid, split):
    name = "r11__%s__%s.npz" % (mid, split)
    p = CACHE / name if (CACHE / name).exists() else PRIOR_CACHE.get(name)
    if p is None:
        return None
    z = np.load(p, allow_pickle=False)
    return {k: z[k] for k in z.files}

if RUN_EVAL:
    runs = discover_runs2()
    print("\nmodels discovered (%d):" % len(runs))
    for k, v in runs.items():
        print("  %-34s trained on %-9s epochs=%s" % (k, v["trained_on"], v["epochs"]))
    missing_expl = [c for c in EXPLORATION if "%s_LSTM_seed99" % c.lower() not in runs]
    if missing_expl:
        print("\nWARNING: no exploration LSTM run for", missing_expl, "-- attach wdn-runs-exploration (notebook 10)")
    for split in TEST_SETS:
        fn = EXPLORATION_FILE.format(code=split) if split in EXPLORATION else (STEAD_FILE if split == "STEAD" else INSTANCE_FILE)
        if find_one(fn, required=False) is None:
            print("\nskipping %s split: %s not attached" % (split, fn)); continue
        print("\n" + "=" * 90 + "\nTEST SPLIT: %s (%s)\n" % (split, CATEGORY[split]) + "=" * 90)
        TS = load_any_test(split); X, Y, meta = TS["X"], TS["Y"], TS["meta"]
        boot = make_boot_indices(TS["clusters"])
        to_score = [(mid, "model", info) for mid, info in runs.items()] + [("baseline_identity", "identity", None), ("baseline_bandpass", "bandpass", None)]
        for mid, kind, info in to_score:
            t0 = time.time(); r = cached(mid, split) if REUSE_CACHE else None
            if r is None:
                if kind == "model":
                    model = load_model(info["path"]); npar = n_params(model); pf = make_predictor("model", model)
                else:
                    npar = 0; pf = make_predictor(kind, band=TS["band"], fs=TS["fs"])
                e = eval_predictor(pf, X, Y)
                g, sin = snr_gain_db(X, Y, pf)
                r = {k: e[k] for k in ("ev_eq", "ev_noise", "mse_eq", "mse_noise", "corr_eq", "resid", "ev_eq_clipped", "ev_noise_clipped")}
                r.update(snr_gain_db=g, snr_in_db=sin, n_params=np.array(npar))
                np.savez_compressed(CACHE / ("r11__%s__%s.npz" % (mid, split)), **r)
                if kind == "model":
                    del model
                del pf
            med_eq = lambda ix, _r=r: float(np.median(_r["ev_eq"][ix])); med_nz = lambda ix, _r=r: float(np.median(_r["ev_noise"][ix]))
            cl_eq = lambda ix, _r=r: float(_r["ev_eq_clipped"][ix].mean()); cl_nz = lambda ix, _r=r: float(_r["ev_noise_clipped"][ix].mean())
            loss_f = lambda ix, _r=r: float((_r["mse_eq"][ix] + _r["mse_noise"][ix]).mean())
            gain_f = lambda ix, _r=r: float(np.median(_r["snr_gain_db"][ix]))
            ci = {k: boot_ci(f, boot) for k, f in (("med_eq", med_eq), ("med_nz", med_nz), ("cl_eq", cl_eq), ("cl_nz", cl_nz), ("loss", loss_f), ("gain", gain_f))}
            al = np.arange(len(X))
            row = dict(model_id=mid, test_split=split, test_category=CATEGORY[split], kind=(info["kind"] if info else "baseline"),
                       bottleneck=(info["bottleneck"] if info else mid.replace("baseline_", "")), trained_on=(info["trained_on"] if info else ""),
                       in_domain=bool(info and info["trained_on"] == split), seed=(info["seed"] if info else ""), n_params=int(r["n_params"]),
                       n_rows=len(X), eval_batch=EVAL_BATCH,
                       median_ev_eq=med_eq(al), median_ev_eq_lo=ci["med_eq"][0], median_ev_eq_hi=ci["med_eq"][1],
                       median_ev_noise=med_nz(al), median_ev_noise_lo=ci["med_nz"][0], median_ev_noise_hi=ci["med_nz"][1],
                       mean_clipped_ev_eq=cl_eq(al), mean_clipped_ev_eq_lo=ci["cl_eq"][0], mean_clipped_ev_eq_hi=ci["cl_eq"][1],
                       mean_clipped_ev_noise=cl_nz(al), mean_clipped_ev_noise_lo=ci["cl_nz"][0], mean_clipped_ev_noise_hi=ci["cl_nz"][1],
                       frac_ev_eq_lt0=float((r["ev_eq"] < 0).mean()), frac_ev_noise_lt0=float((r["ev_noise"] < 0).mean()),
                       test_loss=loss_f(al), test_loss_lo=ci["loss"][0], test_loss_hi=ci["loss"][1],
                       median_snr_gain_db=gain_f(al), median_snr_gain_db_lo=ci["gain"][0], median_snr_gain_db_hi=ci["gain"][1],
                       mean_corr_eq=float(r["corr_eq"].mean()), consistency_residual=float(np.median(r["resid"])),
                       bandpass_hz=("%.2f-%.2f" % TS["band"]) if mid == "baseline_bandpass" else "",
                       runtime_s=(info.get("runtime_s") if info else ""), eval_seconds=round(time.time() - t0, 1))
            SUMMARY.append(row)
            print("  %-34s medEV signal %7.4f  noise %7.4f  loss %.4f  SNR gain %6.2f dB  (%.0fs)"
                  % (mid, row["median_ev_eq"], row["median_ev_noise"], row["test_loss"], row["median_snr_gain_db"], row["eval_seconds"]))
            snr = pd.to_numeric(meta["snr"], errors="coerce").values if "snr" in meta else None
            if snr is not None and np.isfinite(snr).any():
                wb = np.digitize(snr, SNR_EDGES) - 1
                for b in range(len(SNR_EDGES) - 1):
                    ix = np.where(wb == b)[0]
                    if len(ix) >= 5:
                        SNR_ROWS.append(dict(model_id=mid, test_split=split, snr_bin=b, snr_lo=SNR_EDGES[b], snr_hi=SNR_EDGES[b + 1], n=len(ix),
                                             mean_clipped_ev_eq=float(r["ev_eq_clipped"][ix].mean()), mean_clipped_ev_noise=float(r["ev_noise_clipped"][ix].mean()),
                                             median_snr_gain_db=float(np.median(r["snr_gain_db"][ix]))))
        # rows and predictions kept for the figures (exploration splits only)
        if split in EXPLORATION:
            own = "%s_LSTM_seed99" % split.lower()
            show = [m for m in (own, "stead_LSTM_seed99", "instance_LSTM_seed99") if m in runs]
            if own in runs:
                q = cached(own, split)["ev_eq_clipped"]; order = np.argsort(q)
                picks = [int(order[int(f * (len(q) - 1))]) for f in (0.95, 0.50, 0.05)]
                sn = pd.to_numeric(meta["snr"], errors="coerce").values; so = np.argsort(sn)
                section = so[np.linspace(0, len(so) - 1, SECTION_N).astype(int)]
                spec = np.linspace(0, len(X) - 1, min(300, len(X))).astype(int)
                keep = np.unique(np.concatenate([picks, section, spec]))
                store = dict(keep=keep, picks=np.array(picks), pick_labels=np.array(["good (95th pct)", "median", "failure (5th pct)"]),
                             section=section, spec=spec, x=X[keep], y=Y[keep], snr=sn[keep], dt=np.array(TS["dt"]),
                             band=np.array(TS["band"]), shown=np.array(show), ev_own=q[keep])
                for m in show:
                    mdl = load_model(runs[m]["path"]); pe, pn = make_predictor("model", mdl)(X[keep]); del mdl
                    store["pe__" + m] = pe; store["pn__" + m] = pn
                pe, pn = make_predictor("bandpass", band=TS["band"], fs=TS["fs"])(X[keep]); store["pe__baseline_bandpass"] = pe
                np.savez_compressed(RESULTS / ("examples_%s.npz" % split), **store)
                print("  kept %d rows and predictions of %s for the figures" % (len(keep), show + ["baseline_bandpass"]))
        del TS, X, Y, boot
    pd.DataFrame(SUMMARY).to_csv(RESULTS / "table_all_eval.csv", index=False)
    pd.DataFrame(SNR_ROWS).to_csv(RESULTS / "table_ev_vs_snr_all.csv", index=False)
    print("\nwrote table_all_eval.csv (%d rows)" % len(SUMMARY))
    log("11 eval: %d model x split rows over %s" % (len(SUMMARY), TEST_SETS))
else:
    print("evaluation skipped")


models discovered (22):
  f3_LSTM_seed99                     trained on F3        epochs=100
  instance_LSTM_seed99               trained on INSTANCE  epochs=100
  instance_Linear_seed99             trained on INSTANCE  epochs=100
  instance_None_seed99               trained on INSTANCE  epochs=100
  instance_Transformer_seed99        trained on INSTANCE  epochs=100
  instance_attention_seed99          trained on INSTANCE  epochs=100
  pretrained_LSTM                    trained on AUTHORS   epochs=None
  pretrained_Linear                  trained on AUTHORS   epochs=None
  pretrained_None                    trained on AUTHORS   epochs=None
  pretrained_Transformer             trained on AUTHORS   epochs=None
  pretrained_attention               trained on AUTHORS   epochs=None
  stead_LSTM_seed99                  trained on STEAD     epochs=100
  stead_Linear_seed99                trained on STEAD     epochs=100
  stead_None_seed99                  trained on STEAD     epochs=100
  st

## Tables

In [6]:
REF07 = {"instance_LSTM_seed99|STEAD": [0.928545, 0.851725, 0.079252], "instance_Linear_seed99|STEAD": [0.872996, 0.748351, 0.139831], "instance_None_seed99|STEAD": [0.878617, 0.73004, 0.146017], "instance_Transformer_seed99|STEAD": [0.893532, 0.722926, 0.149863], "instance_attention_seed99|STEAD": [0.932825, 0.841868, 0.084289], "pretrained_LSTM|STEAD": [0.974864, 0.923379, 0.047], "pretrained_Linear|STEAD": [0.946019, 0.87348, 0.072566], "pretrained_None|STEAD": [0.946906, 0.874955, 0.072752], "pretrained_Transformer|STEAD": [0.947333, 0.879962, 0.071605], "pretrained_attention|STEAD": [0.974312, 0.918992, 0.048448], "stead_LSTM_seed99|STEAD": [0.9738, 0.9262, 0.044326], "stead_Linear_seed99|STEAD": [0.946633, 0.878191, 0.069545], "stead_None_seed99|STEAD": [0.948189, 0.876525, 0.070804], "stead_Transformer_fixed_seed99|STEAD": [0.977092, 0.925529, 0.04353], "stead_Transformer_seed99|STEAD": [0.953583, 0.884571, 0.066887], "stead_attention_LSTM_seed99|STEAD": [0.974023, 0.929334, 0.044158], "stead_attention_PE_seed99|STEAD": [0.976893, 0.921283, 0.044209], "stead_attention_seed99|STEAD": [0.97558, 0.925343, 0.043941], "stead_hybrid_seed99|STEAD": [0.973833, 0.922999, 0.045087], "baseline_identity|STEAD": [0.544159, -0.987841, 1.000369], "baseline_bandpass|STEAD": [0.836577, 0.018122, 0.504655], "instance_LSTM_seed99|INSTANCE": [0.922024, 0.894037, 0.061311], "instance_Linear_seed99|INSTANCE": [0.887069, 0.858086, 0.09027], "instance_None_seed99|INSTANCE": [0.892706, 0.850841, 0.088616], "instance_Transformer_seed99|INSTANCE": [0.896969, 0.862791, 0.084932], "instance_attention_seed99|INSTANCE": [0.921549, 0.891227, 0.063942], "pretrained_LSTM|INSTANCE": [0.957, 0.915254, 0.051527], "pretrained_Linear|INSTANCE": [0.919555, 0.811732, 0.10642], "pretrained_None|INSTANCE": [0.923663, 0.828729, 0.094474], "pretrained_Transformer|INSTANCE": [0.914674, 0.822647, 0.100687], "pretrained_attention|INSTANCE": [0.956993, 0.916357, 0.050522], "stead_LSTM_seed99|INSTANCE": [0.956141, 0.913294, 0.052715], "stead_Linear_seed99|INSTANCE": [0.913903, 0.827556, 0.09572], "stead_None_seed99|INSTANCE": [0.921397, 0.834184, 0.088726], "stead_Transformer_fixed_seed99|INSTANCE": [0.959576, 0.908751, 0.053725], "stead_Transformer_seed99|INSTANCE": [0.91585, 0.809968, 0.106214], "stead_attention_LSTM_seed99|INSTANCE": [0.95407, 0.910913, 0.056272], "stead_attention_PE_seed99|INSTANCE": [0.960011, 0.914427, 0.050436], "stead_attention_seed99|INSTANCE": [0.957273, 0.911933, 0.052546], "stead_hybrid_seed99|INSTANCE": [0.956447, 0.912799, 0.052196], "baseline_identity|INSTANCE": [0.320554, -0.471789, 1.000721], "baseline_bandpass|INSTANCE": [0.80006, 0.272972, 0.416946]}   # notebook 07: median EV earthquake, median EV noise, test loss

if RUN_TABLES:
    A = read_csv_keep(RESULTS / "table_all_eval.csv")
    # (1) consistency with notebook 07 on the shared STEAD / INSTANCE rows
    chk = []
    for _, r in A.iterrows():
        k = "%s|%s" % (r.model_id, r.test_split)
        if k in REF07:
            ref = REF07[k]
            chk.append(dict(model_id=r.model_id, test_split=r.test_split, median_ev_eq_here=r.median_ev_eq, median_ev_eq_nb07=ref[0],
                            diff_eq=r.median_ev_eq - ref[0], diff_noise=r.median_ev_noise - ref[1], diff_loss=r.test_loss - ref[2]))
    CHK = pd.DataFrame(chk); CHK.to_csv(RESULTS / "check_vs_notebook07.csv", index=False)
    if len(CHK):
        worst = CHK[["diff_eq", "diff_noise", "diff_loss"]].abs().max().max()
        print("check against notebook 07: %d shared rows, largest |difference| = %.2e %s" % (len(CHK), worst,
              "(OK: same scoring)" if worst < 1e-3 else "<- LARGER THAN EXPECTED: read check_vs_notebook07.csv"))
    # (2) transfer matrix of the LSTM: trained on each of the 5 domains x tested on each of the 5 splits
    L = A[(A.bottleneck == "LSTM") & (A.kind != "pretrained") & (A.trained_on.isin(TRAIN_DOMAINS))].copy()
    L.to_csv(RESULTS / "table_lstm_transfer_matrix.csv", index=False)
    for col, lab in (("median_ev_eq", "median EV, signal / earthquake branch"), ("median_ev_noise", "median EV, noise branch"), ("median_snr_gain_db", "median SNR gain (dB)")):
        W = L.pivot_table(index="trained_on", columns="test_split", values=col).reindex(index=TRAIN_DOMAINS, columns=TEST_SETS)
        W.to_csv(RESULTS / ("table_lstm_transfer_matrix_%s.csv" % col))
        print("\nLSTM transfer matrix -- %s (rows: trained on, columns: tested on)" % lab); print(W.round(3).to_string())
    # (3) ranking on each exploration split
    rk = []
    for s in EXPLORATION:
        d = A[A.test_split == s].sort_values("median_ev_eq", ascending=False).reset_index(drop=True)
        d.insert(0, "rank", np.arange(1, len(d) + 1)); rk.append(d)
        own = "%s_LSTM_seed99" % s.lower()
        if own in set(d.model_id):
            r = d[d.model_id == own].iloc[0]; best_other = d[d.model_id != own].iloc[0]
            print("\n%s (%s): in-domain LSTM rank %d of %d | median EV signal %.4f [%.4f, %.4f], noise %.4f, SNR gain %.2f dB | best other: %s (%.4f)"
                  % (s, CATEGORY[s], r["rank"], len(d), r.median_ev_eq, r.median_ev_eq_lo, r.median_ev_eq_hi, r.median_ev_noise, r.median_snr_gain_db,
                     best_other.model_id, best_other.median_ev_eq))
    RANK = pd.concat(rk) if rk else pd.DataFrame(); RANK.to_csv(RESULTS / "table_exploration_ranking.csv", index=False)
    # (4) paired differences on the exploration splits (cluster bootstrap over spatial blocks)
    pr = []
    for s in EXPLORATION:
        own = "%s_LSTM_seed99" % s.lower(); ro = cached(own, s)
        if ro is None:
            continue
        with h5py.File(find_one(EXPLORATION_FILE.format(code=s)), "r") as f:
            te = np.sort(np.where(f["split_id"][:] == 2)[0])
            if MAX_TEST_ROWS:
                te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
            cl = f["source_id"][:][te].astype(str)
        boot = make_boot_indices(cl)
        d = A[(A.test_split == s) & (A.model_id != own)].sort_values("median_ev_eq", ascending=False)
        others = ["stead_LSTM_seed99", "instance_LSTM_seed99", "pretrained_LSTM", "baseline_bandpass", "baseline_identity"]
        others += [m for m in d.model_id if m not in others][:1]                  # + the best other model
        for o in others:
            rb = cached(o, s)
            if rb is None:
                continue
            for metric, key in (("mean_clipped_ev_signal", "ev_eq_clipped"), ("mean_clipped_ev_noise", "ev_noise_clipped"), ("mean_snr_gain_db", "snr_gain_db")):
                dd = ro[key] - rb[key]; lo, hi = boot_ci(lambda ix, _d=dd: float(_d[ix].mean()), boot)
                pr.append(dict(test_split=s, model_a=own, model_b=o, metric=metric, mean_diff=float(dd.mean()), ci_lo=lo, ci_hi=hi,
                               significant_at_95=bool(lo > 0 or hi < 0)))
    PR = pd.DataFrame(pr); PR.to_csv(RESULTS / "table_exploration_paired.csv", index=False)
    if len(PR):
        print("\npaired differences, in-domain LSTM minus other (95 % CI over spatial blocks):")
        print(PR[PR.metric == "mean_clipped_ev_signal"][["test_split", "model_b", "mean_diff", "ci_lo", "ci_hi", "significant_at_95"]].round(4).to_string(index=False))
    S = read_csv_keep(RESULTS / "table_ev_vs_snr_all.csv")
    S[S.test_split.isin(EXPLORATION)].to_csv(RESULTS / "table_exploration_ev_vs_snr.csv", index=False)
    log("11 tables: transfer matrix, ranking, paired, ev vs snr, check vs 07")
else:
    print("tables skipped")

check against notebook 07: 42 shared rows, largest |difference| = 2.52e-05 (OK: same scoring)

LSTM transfer matrix -- median EV, signal / earthquake branch (rows: trained on, columns: tested on)
test_split  STEAD  INSTANCE   VG2D     F3  STR3D
trained_on                                      
STEAD       0.974     0.956  0.001  0.000  0.002
INSTANCE    0.929     0.922  0.024  0.000  0.002
VG2D        0.001    -0.000  0.941  0.871  0.906
F3         -0.010    -0.016  0.212  0.969  0.202
STR3D      -0.001    -0.000  0.891  0.689  0.969

LSTM transfer matrix -- median EV, noise branch (rows: trained on, columns: tested on)
test_split  STEAD  INSTANCE   VG2D     F3  STR3D
trained_on                                      
STEAD       0.926     0.913 -7.521 -9.636 -8.890
INSTANCE    0.852     0.894 -4.570 -6.936 -4.908
VG2D       -0.113     0.030  0.414  0.000  0.022
F3         -0.015     0.094 -5.514  0.647 -4.646
STR3D       0.005     0.280  0.042 -1.792  0.677

LSTM transfer matrix -- media

## Figures

In [7]:
if RUN_FIGURES:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    # notebook 07 style and palette (validated: colour-blind separation passes; the light green / yellow need labels -> every figure has a CSV)
    C_BLUE, C_ORANGE, C_GREEN, C_YELLOW, C_MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e24d8b"
    INK, INK2, GRID, SURF, MUTED = "#0b0b0b", "#52514e", "#e5e4e0", "#fcfcfb", "#9a9a94"
    plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "font.size": 9, "axes.titlesize": 9.5,
                         "axes.labelsize": 9, "axes.edgecolor": INK2, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
                         "text.color": INK, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 8, "ytick.labelsize": 8,
                         "grid.color": GRID, "grid.linewidth": 0.7, "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 1.8,
                         "lines.markersize": 5, "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42, "ps.fonttype": 42})
    GROUP_COLOR = {"pretrained": C_GREEN, "paper": C_BLUE, "new": C_ORANGE, "instance": C_YELLOW, "exploration": C_MAGENTA, "baseline": MUTED}
    GROUP_LABEL = {"pretrained": "authors' pretrained", "paper": "trained on STEAD (paper bottlenecks)", "new": "trained on STEAD (new variants)",
                   "instance": "trained on INSTANCE", "exploration": "trained on exploration data (LSTM)", "baseline": "baselines"}
    def group_of(r):
        if r.kind == "paper" and r.trained_on == "INSTANCE":
            return "instance"
        return r.kind if r.kind in GROUP_COLOR else "baseline"
    def save_fig(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / ("%s.%s" % (name, ext)))
        plt.close(fig); print("  figure: %s (.pdf, .png)" % name)
    DIV = LinearSegmentedColormap.from_list("div", [C_ORANGE, "#efeeea", C_BLUE])      # diverging: worse than nothing <- 0 -> better
    A = read_csv_keep(RESULTS / "table_all_eval.csv")

    # (1) transfer matrix heat maps (LSTM)
    L = A[(A.bottleneck == "LSTM") & (A.kind != "pretrained") & (A.trained_on.isin(TRAIN_DOMAINS))]
    if len(L):
        fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.3))
        for ax, (col, ttl, lim) in zip(axes, (("median_ev_eq", "(a) median EV, signal branch", 1.0), ("median_ev_noise", "(b) median EV, noise branch", 1.0),
                                             ("median_snr_gain_db", "(c) median SNR gain (dB)", None))):
            W = L.pivot_table(index="trained_on", columns="test_split", values=col).reindex(index=TRAIN_DOMAINS, columns=TEST_SETS)
            v = W.to_numpy(float); lim_ = lim if lim else max(1.0, np.nanmax(np.abs(v)))
            ax.imshow(np.clip(v, -lim_, lim_), cmap=DIV, vmin=-lim_, vmax=lim_, aspect="auto")
            for i in range(v.shape[0]):
                for j in range(v.shape[1]):
                    if np.isfinite(v[i, j]):
                        ax.text(j, i, ("%.2f" % v[i, j]) if lim else ("%.1f" % v[i, j]), ha="center", va="center", fontsize=8,
                                color="white" if abs(v[i, j]) > 0.6 * lim_ else INK, fontweight="bold" if TRAIN_DOMAINS[i] == TEST_SETS[j] else "normal")
                    if TRAIN_DOMAINS[i] == TEST_SETS[j]:
                        ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec=INK, lw=1.4))
            ax.set_xticks(range(len(TEST_SETS))); ax.set_xticklabels(TEST_SETS); ax.set_yticks(range(len(TRAIN_DOMAINS))); ax.set_yticklabels(TRAIN_DOMAINS)
            ax.set_xlabel("tested on"); ax.set_title(ttl, loc="left"); ax.spines[:].set_visible(False)
        axes[0].set_ylabel("LSTM trained on")
        fig.suptitle("Final model (LSTM bottleneck) trained on each dataset and tested on every test split. Framed cells: in-domain. "
                     "Orange = worse than doing nothing (EV < 0 or gain < 0 dB).", fontsize=8.5, color=INK2, y=1.02)
        fig.tight_layout(); save_fig(fig, "fig_transfer_matrix_lstm")

    # (2) every model on each exploration split
    E = A[A.test_split.isin(EXPLORATION)]
    if len(E):
        n = E.model_id.nunique()
        fig, axes = plt.subplots(1, len(EXPLORATION), figsize=(5.2 * len(EXPLORATION), 0.24 * n + 1.6), sharex=False)
        axes = np.atleast_1d(axes)
        for ax, s in zip(axes, EXPLORATION):
            d = E[E.test_split == s].sort_values("median_ev_eq")
            if not len(d):
                ax.set_visible(False); continue
            y = np.arange(len(d)); cols = [GROUP_COLOR[group_of(r)] for r in d.itertuples()]
            lo = np.clip(d.median_ev_eq - d.median_ev_eq_lo, 0, None); hi = np.clip(d.median_ev_eq_hi - d.median_ev_eq, 0, None)
            ax.barh(y, d.median_ev_eq, height=0.72, color=cols, edgecolor=SURF, linewidth=1.0, zorder=3)
            ax.errorbar(d.median_ev_eq, y, xerr=[lo, hi], fmt="none", ecolor=INK2, elinewidth=0.8, capsize=1.5, zorder=4)
            labs = []
            for r in d.itertuples():
                labs.append(("$\\bf{%s}$" % r.model_id.replace("_", "\\_")) if r.in_domain else r.model_id)
            ax.set_yticks(y); ax.set_yticklabels(labs, fontsize=7)
            for yi, r in zip(y, d.itertuples()):
                if r.in_domain or r.model_id.startswith("baseline"):
                    ax.annotate("%.3f" % r.median_ev_eq, (max(r.median_ev_eq_hi, r.median_ev_eq), yi), xytext=(4, 0), textcoords="offset points",
                                va="center", fontsize=7.5, color=INK)
            ax.axvline(0, color=INK2, lw=0.8); ax.grid(True, axis="x", zorder=0); ax.set_axisbelow(True)
            ax.set_xlabel("median EV, signal branch (95 % CI)"); ax.set_title("%s test split (%s)" % (s, CATEGORY[s]), loc="left")
            xmin = min(-0.05, float(d.median_ev_eq_lo.min()) - 0.05); ax.set_xlim(max(xmin, -1.5), 1.12)
        handles = [plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in GROUP_LABEL]
        fig.legend(handles, list(GROUP_LABEL.values()), loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
        fig.tight_layout(); save_fig(fig, "fig_exploration_all_models")

    # (3) EV against input SNR on the exploration splits
    S = read_csv_keep(RESULTS / "table_ev_vs_snr_all.csv")
    S = S[S.test_split.isin(EXPLORATION)]
    if len(S):
        fig, axes = plt.subplots(1, len(EXPLORATION), figsize=(5.0 * len(EXPLORATION), 3.6), sharey=True); axes = np.atleast_1d(axes)
        for ax, s in zip(axes, EXPLORATION):
            series = [("%s_LSTM_seed99" % s.lower(), "LSTM trained on %s (in-domain)" % s, C_MAGENTA), ("stead_LSTM_seed99", "LSTM trained on STEAD", C_BLUE),
                      ("instance_LSTM_seed99", "LSTM trained on INSTANCE", C_YELLOW), ("baseline_bandpass", "band-pass", INK2), ("baseline_identity", "identity", MUTED)]
            for mid, lab, colr in series:
                d = S[(S.test_split == s) & (S.model_id == mid)].sort_values("snr_lo")
                if len(d):
                    xc = np.sqrt(d.snr_lo * d.snr_hi)
                    ax.plot(xc, d.mean_clipped_ev_eq, marker="o", color=colr, label=lab, ls="--" if mid.startswith("baseline") else "-", ms=4, lw=1.6,
                            markeredgecolor=SURF, markeredgewidth=0.8, zorder=3)
            ax.axhline(0, color=INK2, lw=0.8); ax.set_xscale("log"); ax.grid(True, zorder=0); ax.set_axisbelow(True)
            ax.set_xlabel("input SNR (amplitude factor of the signal)"); ax.set_title("%s (%s)" % (s, CATEGORY[s]), loc="left")
        axes[0].set_ylabel("mean EV, signal branch (clipped at -1)")
        h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.08))
        fig.tight_layout(); save_fig(fig, "fig_exploration_ev_vs_snr")

    # (4) examples, (5) sections, (6) spectra -- from the kept rows
    EX = {s: dict(np.load(RESULTS / ("examples_%s.npz" % s), allow_pickle=False)) for s in EXPLORATION if (RESULTS / ("examples_%s.npz" % s)).exists()}
    for s, z in EX.items():
        keep = list(z["keep"]); pos = {int(k): i for i, k in enumerate(keep)}; dt = float(z["dt"]); tt = np.arange(z["x"].shape[1]) * dt
        own = "%s_LSTM_seed99" % s.lower()
        fig, ax = plt.subplots(3, 3, figsize=(14, 7), sharex=True)
        for c, (row, lab) in enumerate(zip(z["picks"], z["pick_labels"])):
            i = pos[int(row)]
            ax[0, c].plot(tt, z["x"][i][:, 0], color=INK, lw=0.7); ax[0, c].set_title("%s %s: test row %d, signal EV %.2f, input SNR %.2f"
                                                                                  % (s, lab, row, z["ev_own"][i], z["snr"][i]), loc="left", fontsize=8)
            ax[1, c].plot(tt, z["y"][i][:, 0], color=MUTED, lw=1.6, label="truth")
            ax[1, c].plot(tt, z["pe__" + own][i][:, 0], color=C_MAGENTA, lw=0.9, label="LSTM (%s)" % s)
            if "pe__stead_LSTM_seed99" in z:
                ax[1, c].plot(tt, z["pe__stead_LSTM_seed99"][i][:, 0], color=C_BLUE, lw=0.7, label="LSTM (STEAD)")
            ax[2, c].plot(tt, z["x"][i][:, 0] - z["y"][i][:, 0], color=MUTED, lw=1.6, label="truth")
            ax[2, c].plot(tt, z["pn__" + own][i][:, 0], color=C_MAGENTA, lw=0.9, label="LSTM (%s)" % s)
            for a in ax[:, c]:
                a.grid(True, zorder=0)
        for r_, t_ in enumerate(("input (trace 1 of 3)", "signal", "noise")):
            ax[r_, 0].set_ylabel(t_)
        for a in ax[2]:
            a.set_xlabel("time in window (s)")
        ax[1, 0].legend(fontsize=7, loc="upper right"); fig.tight_layout(); save_fig(fig, "fig_exploration_examples_%s" % s)
    if EX:
        fig, axes = plt.subplots(len(EX), 4, figsize=(14, 3.3 * len(EX)), squeeze=False)
        for r_, (s, z) in enumerate(EX.items()):
            keep = list(z["keep"]); ii = [keep.index(int(k)) for k in z["section"]]; own = "%s_LSTM_seed99" % s.lower(); dt = float(z["dt"])
            panels = (("input", z["x"][ii][:, :, 0]), ("signal truth", z["y"][ii][:, :, 0]), ("LSTM (%s) output" % s, z["pe__" + own][ii][:, :, 0]),
                      ("truth - output", z["y"][ii][:, :, 0] - z["pe__" + own][ii][:, :, 0]))
            clip = np.percentile(np.abs(panels[0][1]), 98)
            for c, (ttl, img) in enumerate(panels):
                axx = axes[r_, c]; axx.imshow(img.T, aspect="auto", cmap="gray", vmin=-clip, vmax=clip, extent=[0, len(ii), img.shape[1] * dt, 0])
                axx.set_title("%s: %s" % (s, ttl), loc="left"); axx.set_xlabel("test window (sorted by input SNR, low -> high)")
            axes[r_, 0].set_ylabel("time in window (s)")
        fig.suptitle("%d test windows side by side (trace 1 of 3), sorted by input SNR: not a continuous seismic section" % SECTION_N, fontsize=8.5, color=INK2, y=1.01)
        fig.tight_layout(); save_fig(fig, "fig_exploration_sections")
        fig, axes = plt.subplots(1, len(EX), figsize=(5.0 * len(EX), 3.5), squeeze=False)
        for axx, (s, z) in zip(axes[0], EX.items()):
            keep = list(z["keep"]); ii = [keep.index(int(k)) for k in z["spec"]]; own = "%s_LSTM_seed99" % s.lower(); dt = float(z["dt"])
            fr = np.fft.rfftfreq(z["x"].shape[1], dt); spec = lambda a: np.abs(np.fft.rfft(a[ii], axis=1)).mean(axis=(0, 2))
            for arr, lab, colr, lw in ((z["x"], "input", INK2, 1.0), (z["y"], "signal truth", MUTED, 2.2), (z["pe__" + own], "LSTM (%s) output" % s, C_MAGENTA, 1.4),
                                        (z.get("pe__stead_LSTM_seed99"), "LSTM (STEAD) output", C_BLUE, 1.0)):
                if arr is not None:
                    axx.semilogy(fr, spec(arr), color=colr, lw=lw, label=lab)
            b = z["band"]; axx.axvspan(b[0], b[1], color=GRID, alpha=0.6, zorder=0, label="band-pass baseline band")
            axx.set_xlabel("frequency (Hz)"); axx.set_title("%s (%s)" % (s, CATEGORY[s]), loc="left"); axx.grid(True, zorder=0)
        axes[0, 0].set_ylabel("mean amplitude spectrum"); h, l = axes[0, 0].get_legend_handles_labels()
        fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.1)); fig.tight_layout(); save_fig(fig, "fig_exploration_spectra")
    log("11 figures written")
else:
    print("figures skipped")

  figure: fig_transfer_matrix_lstm (.pdf, .png)
  figure: fig_exploration_all_models (.pdf, .png)
  figure: fig_exploration_ev_vs_snr (.pdf, .png)
  figure: fig_exploration_examples_VG2D (.pdf, .png)
  figure: fig_exploration_examples_F3 (.pdf, .png)
  figure: fig_exploration_examples_STR3D (.pdf, .png)
  figure: fig_exploration_sections (.pdf, .png)
  figure: fig_exploration_spectra (.pdf, .png)
- 2026-11-27 03:09:25  11 figures written


## Next steps

1. Download `results/` and `figures/` from the version output into `Results/Exploration/Evaluation/` of the submission. The cache folder (`results/cache/`) does not need to be downloaded.
2. Report: lead with the transfer matrix and `fig_exploration_all_models`. Then:
   - give the in-domain LSTM against the STEAD/INSTANCE LSTMs and the band-pass, with paired CIs, from `table_exploration_paired.csv`
   - quote the largest difference from `check_vs_notebook07.csv`
3. Limitations to state:
   - the targets are semi-synthetic, and post-stack signal is not noise-free (VG2D uses a brute stack)
   - F3 and STR3D use synthetic noise
   - VG2D noise windows are mirror-padded
   - one seed
   - the CIs cover the test windows (clustered by spatial block), not training randomness